In [13]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
from datetime import datetime

In [14]:
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=181.57.189.150,1433;"
    "DATABASE=Ventas_Comerssia;"
    "UID=sa;"
    "PWD=P@ssw0rd*;"
)

# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [15]:
fecha_corte = pd.to_datetime("2025-08-31")   

# Rango de análisis
fecha_inicio_2y = fecha_corte - pd.DateOffset(years=2)
fecha_inicio_1y = fecha_corte - pd.DateOffset(years=1)

# Umbrales en días
NUEVO_DIAS = 90          # <= 90 -> Nuevo (por cosecha)
MUY_ACTIVO_DIAS = 120    # <= 120 -> Muy Activo
ACTIVO_DIAS = 300        # 121 - 300 -> Activo
POR_INACTIVAR_DIAS = 360 # 301 - 360 -> Por Inactivar
CHURN_DIAS = 390         # 361 - 390 -> Churn
# > CHURN_DIAS -> Inactivo

In [16]:
df_clientes = pd.read_sql("SELECT Cliente, CosechaFecha FROM Ventas_Comerssia.dbo.Cliente_Perfil", engine)
df_clientes["CosechaFecha"] = pd.to_datetime(df_clientes["CosechaFecha"], errors="coerce")

query_ventas = f"""
SELECT Cliente, Fecha, Venta_Neta AS Venta
FROM Ventas_Comerssia.dbo.Ventas_Unificadas
WHERE Fecha BETWEEN '{fecha_inicio_2y:%Y-%m-%d}' AND '{fecha_corte:%Y-%m-%d}'
"""
# Ejecutar y cargar en DataFrame
df_ventas = pd.read_sql(query_ventas, engine)
df_ventas["Fecha"] = pd.to_datetime(df_ventas["Fecha"], errors="coerce")
df_ventas.head()

,Cliente,Fecha,Venta
0,C51674795,2023-08-31,20359.82
1,C51674795,2023-08-31,25717.67
2,C52959891,2023-08-31,50420.17
3,C1001366330,2023-08-31,236302.52
4,C51674795,2023-08-31,57142.86


In [17]:
# ===============================
# Métricas
# - Última compra (en los 2 años)
# - Total último año (ventas entre fecha_inicio_1y y fecha_corte)
# ===============================
ultima_compra = (
    df_ventas.groupby("Cliente")["Fecha"]
    .max()
    .reset_index(name="UltimaCompra")
)

ventas_ultimo_ano = (
    df_ventas[df_ventas["Fecha"].between(fecha_inicio_1y, fecha_corte)]
    .groupby("Cliente")["Venta"]
    .sum()
    .reset_index(name="TotalUltimoAno")
)

# ===============================
# Unir todo sobre la base de clientes
# ===============================
clientes = df_clientes.merge(ultima_compra, on="Cliente", how="left")
clientes = clientes.merge(ventas_ultimo_ano, on="Cliente", how="left")
clientes["TotalUltimoAno"] = clientes["TotalUltimoAno"].fillna(0)

In [18]:
# ===============================
# Segmento valor (Monetary)
# ===============================
def segmento_valor(x):
    if x > 1_400_000:
        return "Diamante"
    elif 700_000 <= x <= 1_400_000:
        return "Oro"
    elif 300_000 <= x < 700_000:
        return "Plata"
    else:
        return "Bronce"

clientes["Segmento"] = clientes["TotalUltimoAno"].apply(segmento_valor)

# ===============================
# Segmentación de recencia (en días)
# ===============================
def segmento_recencia(row):
    # Nuevo por cosecha
    if pd.notnull(row.get("CosechaFecha")):
        diff_cosecha = (fecha_corte - row["CosechaFecha"]).days
        if diff_cosecha <= NUEVO_DIAS:
            return "Nuevo"
    # Si tuvo compra en el periodo (última compra)
    if pd.notnull(row.get("UltimaCompra")):
        diff_dias = (fecha_corte - row["UltimaCompra"]).days
        if diff_dias <= MUY_ACTIVO_DIAS:
            return "Muy Activo"
        elif diff_dias <= ACTIVO_DIAS:
            return "Activo"
        elif diff_dias <= POR_INACTIVAR_DIAS:
            return "Por Inactivar"
        elif diff_dias <= CHURN_DIAS:
            return "Churn"
        else:
            return "Inactivo"
    # Nunca compró (o fecha inválida)
    return "Inactivo"

clientes["Recencia"] = clientes.apply(segmento_recencia, axis=1)

# ===============================
# Resultado final
# ===============================
clientes_final = clientes[[
    "Cliente", "CosechaFecha", "UltimaCompra", "TotalUltimoAno", "Segmento", "Recencia"
]]

In [19]:
# Mostrar ejemplo
print(clientes_final.head(30))

        Cliente CosechaFecha UltimaCompra  TotalUltimoAno Segmento  \
0     C93069089   2023-02-12          NaT            0.00   Bronce   
1     C43208318   2019-12-25          NaT            0.00   Bronce   
2     C42867517   2019-12-22          NaT            0.00   Bronce   
3   C1232400427   2024-01-19   2024-01-19            0.00   Bronce   
4    C901703815   2023-05-01          NaT            0.00   Bronce   
5   C1037659601   2024-06-16   2024-06-16            0.00   Bronce   
6     C16680439   2022-12-19   2025-01-20       280882.35   Bronce   
7   C1037647639   2020-06-25          NaT            0.00   Bronce   
8     C32721021   2020-10-04          NaT            0.00   Bronce   
9     C21069048   2020-09-12   2024-10-10       315126.05    Plata   
10  C1032432782   2020-04-28          NaT            0.00   Bronce   
11  C1088332709   2022-07-17          NaT            0.00   Bronce   
12  C1098717016   2023-12-08   2023-12-08            0.00   Bronce   
13    C80090687   20

In [20]:
clientes_final.to_excel("segmentacion_clientes.xlsx", index=False)

In [21]:
clientes_final.to_sql("Segmentacion_Clientes", engine, if_exists="replace", index=False)

82